In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

In [ ]:
TASK = Path.cwd().parent
CONFIG = json.loads((TASK.parent/'config.json').read_text())
INPUT, OUTPUT = TASK/'input', TASK/'output'
FIRST, LAST = CONFIG['first_roster_year'], CONFIG['last_publication_year']
YEARS = list(range(1, LAST-FIRST+1))
ASSETS = [('articles.csv', 'ArticleYear'), ('conference-proceedings.csv', 'ConfYear')]

In [ ]:
memberships = pd.read_csv(INPUT/'hire_fields.csv', dtype={'PersonId':'Int64'})
if memberships.empty:
    raise ValueError('No observable assistant-professor hires mapped to the 13 target fields')

In [ ]:
def publication_counts(name, year_column):
    frames = []
    for chunk in pd.read_csv(INPUT/name, usecols=['PersonId', year_column, 'ArticleMatchId'],
                             dtype='string', chunksize=500_000):
        person = pd.to_numeric(chunk.PersonId, errors='coerce').astype('Int64')
        year = pd.to_numeric(chunk[year_column], errors='coerce')
        keep = person.isin(memberships.PersonId) & year.between(FIRST, LAST)
        frames.append(pd.DataFrame({'PersonId':person.loc[keep], 'year':year.loc[keep].astype(int),
                                    'work_id':chunk.loc[keep,'ArticleMatchId']}))
    # separate article and conference IDs, as in prep_aarc
    events = pd.concat(frames, ignore_index=True).drop_duplicates(['PersonId', 'work_id'])
    return events.groupby(['PersonId','year']).size()

In [ ]:
counts = pd.concat([publication_counts(name, year) for name,year in ASSETS])
counts = counts.groupby(level=[0,1]).sum().rename('pubs').reset_index()

In [ ]:
people = memberships[['PersonId','hire_year']].drop_duplicates().set_index('PersonId')
grid = people.reset_index().merge(pd.DataFrame({'career_year':YEARS}), how='cross')
grid['year'] = grid.hire_year + grid.career_year - 1
grid = grid.loc[grid.year.le(LAST)].merge(counts, on=['PersonId','year'], how='left')
grid['pubs'] = grid.pubs.fillna(0).astype(int)
panel = grid.pivot(index='PersonId', columns='career_year', values='pubs').reindex(columns=YEARS)
panel = memberships.merge(panel, left_on='PersonId', right_index=True, validate='many_to_one')
# legacy NSF ID convention for AARC people
panel['dblp_id'] = panel.PersonId.astype(str) + '::' + panel.hire_year.astype(str)
panel = panel[['dblp_id','field_id',*YEARS]].sort_values(['field_id','dblp_id'])

In [ ]:
panel.to_csv(OUTPUT/'secret_aarc_13_field_trajectories.csv', index=False)
display(panel.groupby('field_id').dblp_id.nunique().rename('researchers').reset_index())